In [18]:
import pandas as pd 
import requests, json


In [19]:
# Load full dataset with predictions
energy_df = pd.read_csv("../data/energy_source_with_predictions.csv")
# Load test results with errors
rf_results = pd.read_csv("../data/rf_results.csv")

In [20]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': energy_df.nunique(),
    'Data Type': energy_df.dtypes,
    'Missing Values (Total)': energy_df.isnull().sum(),
    'Missing Values (%)': (energy_df.isnull().sum() / len(energy_df)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

                        Unique Values Data Type  Missing Values (Total)  \
Date                             2922    object                       0   
region_key                         12    object                       0   
ECLAIR                             56   float64                       0   
TX                              31040   float64                       0   
TNSOL                           24575   float64                       0   
FXY                             24707   float64                       0   
DRR                             13292   float64                       0   
year                                8     int64                       0   
population                         91   float64                       0   
FUMEE                              24   float64                       0   
density                            91   float64                       0   
superf                             12   float64                       0   
Consumption              

In [21]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': rf_results.nunique(),
    'Data Type': rf_results.dtypes,
    'Missing Values (Total)': rf_results.isnull().sum(),
    'Missing Values (%)': (rf_results.isnull().sum() / len(rf_results)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

              Unique Values Data Type  Missing Values (Total)  \
Date                   2655    object                       0   
region_name              12    object                       0   
abs_error_rf           6348   float64                       0   
error_rf               6348   float64                       0   
y_pred_rf              6348   float64                       0   
y_true                 6344   float64                       0   
day_of_week               7     int64                       0   
Workday?                  2     int64                       0   
density                  89   float64                       0   
year                      8     int64                       0   
DRR                    3844   float64                       0   
FXY                    5935   float64                       0   
TNSOL                  5922   float64                       0   
TX                     6319   float64                       0   
ECLAIR                   

In [22]:
print(energy_df["region_name"].unique())

['Auvergne-Rhône-Alpes' 'Bourgogne-Franche-Comté' 'Bretagne'
 'Centre-Val de Loire' 'Grand Est' 'Hauts-de-France' 'Normandie'
 'Nouvelle-Aquitaine' 'Occitanie' 'Pays de la Loire'
 "Provence-Alpes-Côte d'Azur" 'Île-de-France']


### Methodological note — error metrics

The **Absolute Percentage Error (APE)** is computed at the observation level as:

$$APE = \\frac{|y_{true} - y_{pred}|}{y_{true} + \\epsilon}$$

where $\\epsilon = 10^{-8}$ is a small constant added to avoid division by zero. 
Its effect is negligible given that consumption values are in the tens of kWh per capita.

**Example:** for $y_{true} = 40.0$ and $y_{pred} = 42.0$:

| Metric | Value |
|:-------|------:|
| Error ($y_{true} - y_{pred}$) | −2.0 |
| Absolute error | 2.0 |
| APE | 0.05 → 5% |

The **regional MAPE** is the mean of all individual APEs for test observations 
belonging to that region:

$$MAPE_{region} = \\frac{1}{n} \\sum_{i=1}^{n} APE_i$$

All error metrics are computed **on the held-out test set only** (`rf_results` derives 
from `X_test` / `y_test`). This ensures we measure generalization to unseen data, 
not in-sample fit.

In [25]:
tooltip_data = rf_results.groupby('region_name').agg(
    mean_mape      = ('ape_rf',       'mean'),
    mean_mae       = ('abs_error_rf', 'mean'),
    mean_predicted = ('y_pred_rf',    'mean'),
    mean_true      = ('y_true',       'mean'),
    mean_bias      = ('error_rf',     'mean'),
    density        = ('density',      'mean'),
    n_obs          = ('y_true',       'count')
).round(3).reset_index()

# Convert to dict keyed by region name — easy to look up in D3
metrics_dict = tooltip_data.set_index('region_name').to_dict(orient='index')

with open('../data/region_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics_dict, ensure_ascii=False, indent=2))

{
  "Auvergne-Rhône-Alpes": {
    "mean_mape": 0.04,
    "mean_mae": 1.8,
    "mean_predicted": 45.427,
    "mean_true": 45.352,
    "mean_bias": -0.076,
    "density": 112.586,
    "n_obs": 584
  },
  "Bourgogne-Franche-Comté": {
    "mean_mape": 0.051,
    "mean_mae": 2.002,
    "mean_predicted": 41.075,
    "mean_true": 40.954,
    "mean_bias": -0.121,
    "density": 58.586,
    "n_obs": 523
  },
  "Bretagne": {
    "mean_mape": 0.045,
    "mean_mae": 1.609,
    "mean_predicted": 36.954,
    "mean_true": 36.506,
    "mean_bias": -0.448,
    "density": 121.641,
    "n_obs": 569
  },
  "Centre-Val de Loire": {
    "mean_mape": 0.049,
    "mean_mae": 1.914,
    "mean_predicted": 39.756,
    "mean_true": 39.849,
    "mean_bias": 0.093,
    "density": 65.3,
    "n_obs": 324
  },
  "Grand Est": {
    "mean_mape": 0.042,
    "mean_mae": 1.797,
    "mean_predicted": 44.569,
    "mean_true": 44.519,
    "mean_bias": -0.05,
    "density": 96.29,
    "n_obs": 581
  },
  "Hauts-de-France": {
  

In [29]:
viz_b_data = rf_results[['y_true', 'y_pred_rf', 'abs_error_rf', 'ape_rf', 'region_name']].copy()
viz_b_data = viz_b_data.round(3)

viz_b_data.to_json('../data/rf_scatter.json', orient='records')
print(f"Total points: {len(viz_b_data)}")
print(f"y_true:  {viz_b_data['y_true'].min():.2f} — {viz_b_data['y_true'].max():.2f}")
print(f"y_pred:  {viz_b_data['y_pred_rf'].min():.2f} — {viz_b_data['y_pred_rf'].max():.2f}")
print(f"n regions: {viz_b_data['region_name'].nunique()}")

Total points: 6348
y_true:  18.96 — 74.95
y_pred:  21.11 — 71.49
n regions: 12


In [27]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Perfect prediction line range
min_val = min(rf_results['y_true'].min(), rf_results['y_pred_rf'].min())
max_val = max(rf_results['y_true'].max(), rf_results['y_pred_rf'].max())

fig = px.scatter(
    rf_results,
    x='y_true',
    y='y_pred_rf',
    color='region_name',
    hover_data={
        'y_true':      ':.2f',
        'y_pred_rf':   ':.2f',
        'abs_error_rf':':.2f',
        'ape_rf':      ':.3f',
        'region_name': True
    },
    labels={
        'y_true':    'Actual consumption (kWh/capita)',
        'y_pred_rf': 'Predicted consumption (kWh/capita)',
        'region_name': 'Region'
    },
    opacity=0.6,
    title='Predicted vs actual energy consumption per capita — Random Forest'
)

# Perfect prediction diagonal
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    name='Perfect prediction',
    hoverinfo='skip'
))

fig.update_layout(
    width=800, height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Georgia, serif', size=12),
    legend=dict(title='Region', font=dict(size=10)),
    xaxis=dict(showgrid=True, gridcolor='#eee', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='#eee', zeroline=False),
)

fig.show()

# Save as HTML for your demo
fig.write_html("../visualizations/vizB.html")